# 03: Threshold Optimization

Grid search to find optimal green/red thresholds for regime classification.

**Goal:** Maximize stress detection accuracy while minimizing false positives.


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import itertools

from collectors.cbk_collector import CBKCollector
from collectors.cba_collector import CBACollector
from engine.regime_engine import RegimeEngine
from config import MARKETS

print("Threshold optimization starting...")


## 1. Load Historical Data

In [ ]:
# Fetch data
cbk = CBKCollector()
cba = CBACollector()

start = datetime.now() - timedelta(days=730)

kes_data = cbk.get_rates(currency='USD', start_date=start)
azn_data = cba.get_historical_rates(start_date=start, end_date=datetime.now(), currency='USD')

# Calculate volatility
nairobi_engine = RegimeEngine(MARKETS['nairobi'])
baku_engine = RegimeEngine(MARKETS['baku'])

kes_data['volatility'] = kes_data['kes_per_usd'].rolling(30).apply(
    lambda x: nairobi_engine.calculate_fx_volatility(x)
)

print(f"KES data: {len(kes_data)} points")
print(f"AZN data: {len(azn_data)} points")


## 2. Define Known Stress Events

In [ ]:
# Historical stress periods (ground truth)
stress_events = {
    'nairobi': [
        ('2022-08-01', '2022-09-15', 'Post-Election'),
        ('2023-01-01', '2023-04-01', 'Rate Hikes'),
        ('2023-10-01', '2023-12-01', 'EM Stress'),
    ],
    'baku': [
        ('2022-02-01', '2022-04-01', 'Ukraine War'),
        ('2022-12-01', '2023-02-01', 'Oil Price Shock'),
    ]
}

def is_stress_period(date, market):
    # Check if date falls in any stress period
    for start, end, _ in stress_events.get(market, []):
        if pd.to_datetime(start) <= date <= pd.to_datetime(end):
            return True
    return False

# Label data
kes_data['is_stress'] = kes_data['date'].apply(lambda x: is_stress_period(x, 'nairobi'))

print("Stress periods labeled")
print(kes_data['is_stress'].value_counts())


## 3. Grid Search Optimization

In [ ]:
def evaluate_thresholds(data, vol_col, green_thr, red_thr):
    # Classify all days with given thresholds
    classifications = []
    
    for _, row in data.iterrows():
        vol = row[vol_col]
        if pd.isna(vol):
            continue
            
        if vol < green_thr:
            regime = 'Risk-On'
            is_alert = False
        elif vol > red_thr:
            regime = 'Instability'
            is_alert = True
        else:
            regime = 'Defensive'
            is_alert = False
        
        classifications.append({
            'date': row['date'],
            'actual_stress': row['is_stress'],
            'predicted_alert': is_alert,
            'regime': regime,
            'volatility': vol
        })
    
    df = pd.DataFrame(classifications)
    
    # Calculate metrics
    tp = ((df['actual_stress'] == True) & (df['predicted_alert'] == True)).sum()
    fp = ((df['actual_stress'] == False) & (df['predicted_alert'] == True)).sum()
    tn = ((df['actual_stress'] == False) & (df['predicted_alert'] == False)).sum()
    fn = ((df['actual_stress'] == True) & (df['predicted_alert'] == False)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(df)
    
    return {
        'green': green_thr,
        'red': red_thr,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'alert_rate': df['predicted_alert'].mean()
    }

# Grid search
green_range = np.arange(0.02, 0.06, 0.005)
red_range = np.arange(0.05, 0.12, 0.005)

results = []
for green, red in itertools.product(green_range, red_range):
    if green >= red:
        continue
    result = evaluate_thresholds(kes_data, 'volatility', green, red)
    results.append(result)

results_df = pd.DataFrame(results)
print(f"Tested {len(results_df)} threshold combinations")

# Best by F1 score
best_f1 = results_df.loc[results_df['f1'].idxmax()]
print(f"\nBest F1 Score: {best_f1['f1']:.3f}")
print(f"Green: {best_f1['green']:.1%}, Red: {best_f1['red']:.1%}")


## 4. Visualize Optimization Surface

In [ ]:
# Create heatmap
pivot_f1 = results_df.pivot(index='green', columns='red', values='f1')

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# F1 Score heatmap
sns.heatmap(pivot_f1, annot=False, cmap='RdYlGn', ax=axes[0,0], cbar_kws={'label': 'F1 Score'})
axes[0,0].set_title('F1 Score by Thresholds', fontweight='bold')
axes[0,0].set_xlabel('Red Threshold (Instability)')
axes[0,0].set_ylabel('Green Threshold (Risk-On)')

# Precision vs Recall
scatter = axes[0,1].scatter(results_df['recall'], results_df['precision'], 
                           c=results_df['f1'], cmap='viridis', alpha=0.6)
axes[0,1].set_xlabel('Recall (Sensitivity)')
axes[0,1].set_ylabel('Precision')
axes[0,1].set_title('Precision-Recall Trade-off', fontweight='bold')
axes[0,1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0,1], label='F1 Score')

# Alert rate vs Accuracy
axes[1,0].scatter(results_df['alert_rate'] * 100, results_df['accuracy'] * 100, 
                 c=results_df['f1'], cmap='plasma', alpha=0.6)
axes[1,0].set_xlabel('Alert Rate (%)')
axes[1,0].set_ylabel('Accuracy (%)')
axes[1,0].set_title('Alert Rate vs Accuracy', fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

# ROC-like curve
axes[1,1].plot(results_df['fp'] / (results_df['fp'] + results_df['tn']), 
              results_df['tp'] / (results_df['tp'] + results_df['fn']),
              'b.', alpha=0.3)
axes[1,1].set_xlabel('False Positive Rate')
axes[1,1].set_ylabel('True Positive Rate')
axes[1,1].set_title('ROC Space', fontweight='bold')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].plot([0, 1], [0, 1], 'k--', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/threshold_optimization_heatmap.png', dpi=150)
plt.show()

print("Saved heatmap to data/processed/threshold_optimization_heatmap.png")


## 5. Compare Market-Specific Thresholds

In [ ]:
# Optimize for each market
markets = {
    'nairobi': (kes_data, 'volatility', MARKETS['nairobi']),
    'baku': (azn_data, 'azn_per_unit', MARKETS['baku'])
}

optimal_thresholds = {}

for market_name, (data, col, config) in markets.items():
    if data.empty:
        continue
        
    print(f"\nOptimizing {market_name}...")
    
    # Quick optimization
    best_score = 0
    best_params = None
    
    for green in np.arange(0.025, 0.055, 0.005):
        for red in np.arange(0.055, 0.10, 0.005):
            if green >= red:
                continue
            
            # Simple scoring: maximize F1
            result = evaluate_thresholds(data, col, green, red) if market_name == 'nairobi' else None
            
            if result and result['f1'] > best_score:
                best_score = result['f1']
                best_params = (green, red)
    
    if best_params:
        optimal_thresholds[market_name] = {
            'green': best_params[0],
            'red': best_params[1],
            'f1': best_score
        }
        print(f"  Optimal: Green={best_params[0]:.1%}, Red={best_params[1]:.1%}, F1={best_score:.3f}")

# Summary table
summary = pd.DataFrame(optimal_thresholds).T
print("\n=== Optimal Thresholds by Market ===")
print(summary)


## 6. Export Recommendations

In [ ]:
# Save recommendations
recommendations = {
    'nairobi': {
        'fx_vol_green': optimal_thresholds.get('nairobi', {}).get('green', 0.04),
        'fx_vol_red': optimal_thresholds.get('nairobi', {}).get('red', 0.08),
        'rationale': 'Optimized for Kenya election and rate hike cycles'
    },
    'baku': {
        'fx_vol_green': optimal_thresholds.get('baku', {}).get('green', 0.03),
        'fx_vol_red': optimal_thresholds.get('baku', {}).get('red', 0.06),
        'rationale': 'Optimized for oil price shock sensitivity'
    }
}

import json
with open('../data/processed/optimal_thresholds.json', 'w') as f:
    json.dump(recommendations, f, indent=2)

print("Saved: data/processed/optimal_thresholds.json")
print("\nRecommendations:")
print(json.dumps(recommendations, indent=2))
